## 6. RQ4: Moderation by therapeutic area

Strata are the six top-level MeSH disease categories built by `prep_06_mesh_areas.py`.
The residual `Other` category is retained as a control level but **not interpreted**,
66% of it is MeSH-mapped outside the five named categories and 34% is unmapped.


## 1. Environment and data access

In [ ]:
# Colab: install what the base image lacks
!pip install -q lifelines shap pyarrow 2>/dev/null
import pandas as pd, numpy as np, math
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 60)
print("pandas", pd.__version__)

In [ ]:
# Mount Google Drive and point at the analysis-ready extract
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/DBA_Capstone/analysis_ready'   # adjust if needed
import os; print(sorted(os.listdir(DATA)))

### Local alternative

If running outside Colab, point `DATA` at the copy versioned in this repository --
the analysis-ready extract is 41 MB and is committed, so nothing needs downloading:

```python
DATA = "data/analysis_ready"
```

## 2. Load and build the analytic table

In [ ]:
trials = pd.read_parquet(f"{DATA}/ctgov_trials.parquet")
ob     = pd.read_parquet(f"{DATA}/orangebook_ip.parquet")
reg    = pd.read_parquet(f"{DATA}/drugsatfda_regulatory.parquet")
print("trials", trials.shape, "| orangebook", ob.shape, "| regulatory", reg.shape)

# Study frame: Phase II/III drug trials, start 2008-2019, resolved outcome.
#
# The window closes at 2019, not at the extract date. Resolution is still accruing
# after that -- 92.6% of 2008 starts have resolved against 34.8% of 2023 starts --
# and because terminated trials resolve quickly while completed ones take years,
# the recent cohorts are enriched for terminations purely by observation time.
# Running to 2023 would put that right-censoring artefact into the event rate and
# report it as a trend. For the same reason start_year is never a predictor.
frame = trials[(trials.start_year.between(2008, 2019)) & (trials.is_resolved == 1)].copy()
print(f"analytic frame N = {len(frame):,}")
print(f"termination rate  = {frame.is_terminated.mean():.4f}")
print(f"industry subset   = {int(frame.is_industry.sum()):,}")

In [ ]:
# Sponsor-level history computed with PAST-ONLY information (prevents temporal leakage)
frame = frame.sort_values("start_date")
g = frame.groupby("lead_sponsor")
frame["sponsor_prior_trials"] = g.cumcount()
frame["sponsor_prior_terminations"] = (
    g["is_terminated"].cumsum() - frame["is_terminated"])
frame["sponsor_prior_completion_rate"] = np.where(
    frame.sponsor_prior_trials > 0,
    1 - frame.sponsor_prior_terminations / frame.sponsor_prior_trials, np.nan)
frame[["lead_sponsor","start_date","sponsor_prior_trials","sponsor_prior_completion_rate"]].tail(5)

## 3. RQ1: Do termination rates differ by sponsor IP strength?

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency

# IP strength is the stored composite index, not a sponsor-scale stand-in.
# prep_11_ip_index.py builds it from the Orange Book drug-level components,
# winsorised at the 1st/99th percentiles and standardised within start-year cohorts
# so an older patent does not look stronger through exposure time alone.
#
# Sponsor prior-trial count was used here previously. It is a measure of firm size,
# and firm size is the confounder this question has to hold constant, so it cannot
# also serve as the exposure.
ip = pd.read_parquet(f"{DATA}/ip_index.parquet")
frame = frame.merge(ip[["nct_id", "ip_index", "ip_strong", "ip_tertile_extreme"]],
                    on="nct_id", how="left")

# A trial whose drug carries no listed patent is UNOBSERVED on the index, not weak.
# Coding it weak would manufacture the finding, so it leaves the RQ1 subsample --
# and the exclusion is reported, because it is demonstrably not ignorable.
obs = frame.ip_strong.notna()
z_sel, p_sel = proportions_ztest(
    [int(frame.loc[obs, "is_terminated"].sum()),
     int(frame.loc[~obs, "is_terminated"].sum())],
    [int(obs.sum()), int((~obs).sum())])
print(f"index computable for {obs.sum():,} of {len(frame):,} trials")
print(f"  computable terminate at {frame.loc[obs, 'is_terminated'].mean():.4f}, "
      f"not computable at {frame.loc[~obs, 'is_terminated'].mean():.4f} "
      f"(z = {z_sel:.2f}, p = {p_sel:.2e})")

rq1 = frame[obs].copy()
rq1["ip_strong"] = rq1.ip_strong.astype(int)
ct = pd.crosstab(rq1.ip_strong, rq1.is_terminated)
print(ct)

cnt = [ct.loc[1, 1], ct.loc[0, 1]]; nobs = [ct.loc[1].sum(), ct.loc[0].sum()]
z, p = proportions_ztest(cnt, nobs)
chi2, pchi, dof, _ = chi2_contingency(ct)
v = math.sqrt(chi2 / (ct.values.sum() * min(ct.shape[0] - 1, ct.shape[1] - 1)))
print(f"\nstrong-IP termination = {cnt[0]/nobs[0]:.4f}   weak-IP = {cnt[1]/nobs[1]:.4f}")
print(f"z = {z:.4f}, p = {p:.3e}")
print(f"chi2 = {chi2:.2f}, p = {pchi:.3e}, Cramer's V = {v:.4f}")

## 4. RQ2: Predicting termination

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Design predictors (always available)
DESIGN = ["is_randomized","who_masked_n","n_arms","n_locations","n_countries",
          "is_multinational","elig_criteria_lines","n_primary_outcomes",
          "n_secondary_outcomes","is_phase3","n_interventions","is_biologic"]
SPONSOR = ["is_industry","n_collaborators","sponsor_prior_trials",
           "sponsor_prior_completion_rate"]

# IP components enter RQ2/RQ3 SEPARATELY, not as the composite index.
# Dichotomising is only done for RQ1, where the two-proportion test requires it;
# here the model has the degrees of freedom to use each dimension.
IP_SCALE_FREE = ["citations_per_patent","family_size","family_n_countries",
                 "assignee_pharma_share","ob_term_remaining_yrs"]
IP_SCALE      = ["assignee_portfolio_size","ob_n_patents","ob_n_primary_patents"]
IP_AMBIGUOUS  = ["ob_evergreen_ratio"]     # sign is ambiguous - never in the index
REGULATORY    = ["fda_any_priority","fda_has_orphan","fda_is_nme"]

# NOTE: enrollment is deliberately absent - it records ACTUAL enrollment and
# therefore encodes the outcome (see README).
CANDIDATES = DESIGN + SPONSOR + IP_SCALE_FREE + IP_SCALE + IP_AMBIGUOUS + REGULATORY
FEATS = [c for c in CANDIDATES if c in frame.columns]
missing = [c for c in CANDIDATES if c not in frame.columns]
print(f"using {len(FEATS)} predictors")
if missing:
    print(f"PENDING (not yet joined): {missing}")

# TEMPORAL split - reflects real deployment, not random shuffling
tr = frame[frame.start_year <= 2015]
te = frame[frame.start_year >= 2016]
Xtr, ytr = tr[FEATS].fillna(-1), tr.is_terminated
Xte, yte = te[FEATS].fillna(-1), te.is_terminated
print(f"train {len(tr):,} (2008-2015)   test {len(te):,} (2016-2019)")

lr = make_pipeline(StandardScaler(),
      LogisticRegression(penalty="l1", solver="liblinear", C=0.5,
                         class_weight="balanced", max_iter=2000)).fit(Xtr, ytr)
gb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06,
                                    class_weight="balanced", random_state=42).fit(Xtr, ytr)
for name, m in [("L1 logistic", lr), ("Gradient boosting", gb)]:
    pr = m.predict_proba(Xte)[:,1]
    print(f"{name:20s} AUC={roc_auc_score(yte,pr):.4f}  Brier={brier_score_loss(yte,pr):.4f}")
print("\nLeakage-free baseline to beat: GB AUC 0.676 / LR AUC 0.600")

In [ ]:
# SHAP attribution for the boosted model
import shap
expl = shap.TreeExplainer(gb)
sv = expl.shap_values(Xte.sample(min(2000, len(Xte)), random_state=0))
shap.summary_plot(sv, Xte.sample(min(2000, len(Xte)), random_state=0), max_display=15)

## 5. RQ3: Time to primary completion

In [ ]:
from lifelines import AalenJohansenFitter, CoxPHFitter

# COMPETING RISKS, not plain survival.
# A terminated trial can never reach primary completion, so it is NOT censored --
# it is the competing event. Censoring it would assume completion would have
# happened given more time, which is false, and biases the hazard ratio toward
# the hypothesis. Only trials still active at the extract date are censored.
surv = frame[frame.months_to_primary_completion.notna() &
             (frame.months_to_primary_completion > 0)].copy()
surv["status"] = 0                                   # 0 = censored (still active)
surv.loc[surv.is_terminated == 0, "status"] = 1       # 1 = completed
surv.loc[surv.is_terminated == 1, "status"] = 2       # 2 = terminated (competing)
print(surv.status.value_counts().rename({0:"censored",1:"completed",2:"terminated"}))

# Cumulative incidence: Aalen-Johansen, NOT Kaplan-Meier.
# KM overstates incidence under competing risks because it censors the competitor.
ax = None
for lvl, lab in [(1, "Strong IP"), (0, "Weak IP")]:
    s = surv[surv.ip_strong == lvl]
    ajf = AalenJohansenFitter(calculate_variance=False)
    ajf.fit(s.months_to_primary_completion, s.status, event_of_interest=1)
    ax = ajf.plot(ax=ax, label=lab)
ax.set_xlabel("Months from trial start")
ax.set_ylabel("Cumulative incidence of primary completion")
ax.set_xlim(0, 120)

In [ ]:
# Cause-specific hazards: one model per cause. For the completion model the
# competing event is censored *for that cause only*, which is the correct
# cause-specific construction, and the termination model is reported alongside
# so both effects are visible rather than one being inferred from the other.
#
# enrollment is NOT a covariate: the registry records achieved rather than
# planned enrollment, so it encodes the outcome (see README).
COVARS = [c for c in ["ip_strong", "is_phase3", "n_locations", "is_industry",
                      "is_biologic", "sponsor_prior_trials"] if c in surv.columns]

for cause, name in [(1, "completion"), (2, "termination")]:
    cx = surv[["months_to_primary_completion", "status"] + COVARS].dropna().copy()
    cx["event"] = (cx.status == cause).astype(int)     # cause-specific indicator
    cx = cx.drop(columns="status")
    cph = CoxPHFitter().fit(cx, duration_col="months_to_primary_completion",
                            event_col="event")
    print(f"\n===== cause-specific hazard: {name} =====")
    cph.print_summary(decimals=4)
    print("concordance:", round(cph.concordance_index_, 4))
    # at this N the PH test rejects on trivial departures; stratify if so
    cph.check_assumptions(cx, p_value_threshold=0.01, show_plots=False)

### RQ3, machine-learning comparison

Aalen-Johansen and cause-specific Cox are the interpretable baseline. A Random Survival
Forest is fitted alongside them as the learned counterpart, mirroring the structure of RQ2
(penalised logistic as baseline, gradient boosting as the learner).

The forest is fitted **cause-specifically**: for the completion model the competing event is
censored for that cause only. `RandomSurvivalForest` handles right-censoring but has no native
competing-risks mode, so this is the correct construction rather than a shortcut.

The comparison answers a specific question: does a model free to find non-linearities and
interactions discriminate better than the proportional-hazards specification? If it does not,
that supports reporting the Cox coefficients as the finding.


In [ ]:
# pip install scikit-survival  (not in the Colab base image)
import numpy as np
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

CAUSE = 1                      # 1 = completion, 2 = termination
cx = surv[["months_to_primary_completion", "status"] + COVARS].dropna().copy()
cx["event"] = (cx.status == CAUSE).astype(bool)      # cause-specific indicator

# temporal split, consistent with RQ2 - never a random split
yr = frame.loc[cx.index, "start_year"]
tr_i, te_i = yr <= 2015, yr >= 2016
X_tr, X_te = cx.loc[tr_i, COVARS].astype(float), cx.loc[te_i, COVARS].astype(float)
y_tr = Surv.from_arrays(cx.loc[tr_i, "event"], cx.loc[tr_i, "months_to_primary_completion"])
y_te = Surv.from_arrays(cx.loc[te_i, "event"], cx.loc[te_i, "months_to_primary_completion"])
print(f"train {len(X_tr):,} (2008-2015)   test {len(X_te):,} (2016-2019)")

rsf = RandomSurvivalForest(n_estimators=300, min_samples_leaf=25,
                           max_features="sqrt", n_jobs=-1,
                           random_state=42).fit(X_tr, y_tr)
c_rsf = concordance_index_censored(cx.loc[te_i, "event"],
                                   cx.loc[te_i, "months_to_primary_completion"],
                                   rsf.predict(X_te))[0]

# the Cox baseline on the same split, so the comparison is like for like
cox_cols = ["months_to_primary_completion", "event"] + COVARS
cph_cmp = CoxPHFitter().fit(cx.loc[tr_i, cox_cols].assign(event=cx.loc[tr_i,"event"].astype(int)),
                            duration_col="months_to_primary_completion", event_col="event")
c_cox = concordance_index_censored(cx.loc[te_i, "event"],
                                   cx.loc[te_i, "months_to_primary_completion"],
                                   -cph_cmp.predict_partial_hazard(cx.loc[te_i, COVARS]).values)[0]

print(f"\nout-of-sample concordance (completion)")
print(f"  cause-specific Cox      {c_cox:.4f}")
print(f"  Random Survival Forest  {c_rsf:.4f}")
print(f"  difference              {c_rsf - c_cox:+.4f}")

imp = pd.Series(rsf.feature_importances_ if hasattr(rsf, "feature_importances_")
                else np.zeros(len(COVARS)), index=COVARS).sort_values(ascending=False)
print("\nforest variable ranking:"); print(imp.round(4).to_string())

## 6. RQ4: Moderation by therapeutic area

In [ ]:
# Therapeutic-area strata come from the MeSH mapping built in prep_06_mesh_areas.py
# (AACT browse_conditions ancestors -> top-level MeSH disease categories,
#  priority-assigned so strata are mutually exclusive). Already a column on the frame.
print(frame.therapeutic_area.value_counts().to_string())

from statsmodels.stats.proportion import proportion_confint
rows = []
for a, g_ in frame.groupby("therapeutic_area"):
    lo, hi = proportion_confint(int(g_.is_terminated.sum()), len(g_), method="wilson")
    rows.append({"area": a, "n": len(g_), "p_term": round(g_.is_terminated.mean(), 4),
                 "ci_low": round(lo, 4), "ci_high": round(hi, 4),
                 "margin": round((hi - lo) / 2, 4), "meets_737": len(g_) >= 737})
strata = pd.DataFrame(rows).sort_values("n", ascending=False)
display(strata)
assert strata.meets_737.all(), "a stratum falls below the RQ4 requirement of 737"
print("\nAll strata meet the n >= 737 requirement.")

In [ ]:
# moderated logistic regression: IP x therapeutic area
m0 = smf.logit("is_terminated ~ ip_strong + C(therapeutic_area) + is_phase3 + is_industry",
               data=frame).fit(disp=0)
m1 = smf.logit("is_terminated ~ ip_strong * C(therapeutic_area) + is_phase3 + is_industry",
               data=frame).fit(disp=0)
lr_stat = 2 * (m1.llf - m0.llf)
from scipy.stats import chi2 as chi2dist
df_diff = m1.df_model - m0.df_model
print(f"Likelihood ratio test: chi2 = {lr_stat:.3f}, df = {df_diff:.0f}, "
      f"p = {chi2dist.sf(lr_stat, df_diff):.4f}")
print(m1.summary())

### RQ4, machine-learning corroboration

The moderated logistic regression imposes the interactions it tests. Gradient-boosted trees find
them without being told, and SHAP interaction values decompose a prediction into main effects
and pairwise terms, so the ranking of feature pairs can be read off directly.

Agreement between the parametric coefficients and this ranking strengthens a moderation claim.
Disagreement says the parametric form imposed structure the data does not support. This matters
for RQ4 because interaction tests are lower-powered than main-effect tests, so a single omnibus
statistic is thin evidence on its own.


In [ ]:
import shap, numpy as np

# reuses the gradient-boosted model already fitted for RQ2; nothing new is trained
sample = te.sample(min(2000, len(te)), random_state=0)
iv = shap.TreeExplainer(gb).shap_interaction_values(sample[FEATS].fillna(-1))

m = np.abs(iv).mean(0)          # mean |interaction| per feature pair
np.fill_diagonal(m, 0)          # diagonal holds main effects, not interactions
pairs = sorted(((FEATS[i], FEATS[j], m[i, j])
                for i in range(len(FEATS)) for j in range(i + 1, len(FEATS))),
               key=lambda x: -x[2])

print(f"SHAP interaction values on {len(sample):,} held-out trials\n")
print("strongest feature pairs overall:")
for a, b, v in pairs[:8]:
    print(f"  {v:.5f}   {a} x {b}")

ip_pairs = [p for p in pairs if any(k in p[0] + p[1] for k in ("ob_", "ip_", "assignee", "family"))]
print("\nstrongest pairs involving an IP feature:")
for a, b, v in ip_pairs[:5]:
    print(f"  {v:.5f}   {a} x {b}")

# compare against the parametric interaction terms from the moderation model above:
# a pair ranking high here but non-significant there (or the reverse) is worth reporting.